In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [11]:
df = pd.read_csv('train.csv')
df.sample()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
344,345,0,2,"Fox, Mr. Stanley Hubert",male,36.0,0,0,229236,13.0,NaN,S


# Drop unnecessary columns

In [12]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [13]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [14]:
df.sample()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
22,1,3,female,15.0,0,0,8.0292,Q


# Imputations

In [15]:
trf1 = ColumnTransformer([ ('impute_age' , SimpleImputer() , [2]) , 
                          ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]) ],  remainder='passthrough')

# one hot encoding

In [16]:
trf2 = ColumnTransformer( [ ('ohe_sex_embarked' , OneHotEncoder(sparse_output=False,handle_unknown='ignore'), [1,6]) ]
                         , remainder='passthrough')

# Scaling

In [17]:
trf3 = ColumnTransformer([ ('scale',MinMaxScaler(),slice(0,10)) ])

# train the model

In [19]:
trf5 = DecisionTreeClassifier()

# Pipeline

In [23]:
# pipe = Pipeline([ ('trf1',trf1), ('trf2',trf2), ('trf3',trf3), ('trf4',trf4), ('trf5',trf5) ]) OR
pipe = make_pipeline(trf1,trf2,trf3,trf5)
pipe.fit(X_train,y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [24]:
y_pred = pipe.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6256983240223464

In [25]:
# export 
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))